# 01: Data Ingestion, Ben Graham Normalization & Non-IID Hospital Partitioning

This notebook demonstrates:
1. Loading the APTOS 2019 Blindness Detection dataset.
2. Ben Graham's local-average color-subtraction normalization.
3. Simulating 5 heterogeneous, non-IID hospitals using Dirichlet(alpha=0.5) distribution.
4. Computing the Heterogeneity Score (H) via Jensen-Shannon Divergence.


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data_preprocessing import ben_graham_normalize, DR_CLASSES, DR_DESCRIPTIONS
from src.hospital_partition import partition_dataset_dirichlet, save_splits, load_splits


In [ ]:
df = pd.read_csv('../data/aptos/train.csv')
print(f'Total dataset records: {len(df)}')
print(df['diagnosis'].value_counts().sort_index())


### Ben Graham Illumination Normalization
Fundus photographs often exhibit non-uniform lighting and scanner variations across hospital sites. Ben Graham normalization accentuates retinal lesions (hemorrhages, exudates, microaneurysms) while suppressing illumination artifacts.


In [ ]:
sample_img_id = df.iloc[0]['id_code']
sample_path = f'../data/aptos/train_images/{sample_img_id}.png'
raw_rgb = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
normalized_rgb = ben_graham_normalize(raw_rgb, img_size=224, sigma=10)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(raw_rgb)
axes[0].set_title('Raw Fundus Photography')
axes[0].axis('off')
axes[1].imshow(normalized_rgb)
axes[1].set_title('Ben Graham Normalized (σ=10)')
axes[1].axis('off')
plt.tight_layout()
plt.show()


### 5 Non-IID Hospitals Dirichlet Partitioning & Jensen-Shannon Heterogeneity (H)


In [ ]:
splits, summary_df = partition_dataset_dirichlet(df, num_hospitals=5, alpha=0.5, seed=42)
save_splits(splits, '../artifacts/splits.json')
summary_df
